# 5. Validação: os detectores de risco são confiáveis?

**Pergunta:** quando o código diz "esta skill tem `curl | sh`" ou "esta skill lida com segredos", ele está certo?

**Por que este notebook existe:** todos os números do notebook 04 (6,4% de gravidade alta, 17,3% de variantes que acrescentam capacidade, McNemar) vêm de **expressões regulares**, que procuram palavras no texto. Uma regex não entende contexto:

| Trecho | A regex marca | Na verdade é |
|---|---|---|
| "Instale com `curl -sSL https://… \| bash`" | pipe_to_shell | **instrução real** |
| "Nunca coloque a API key no código" | secrets | só uma **menção** (é até um conselho de segurança) |
| "o arquivo `.env.example` lista as variáveis" | secrets | menção, sem acesso a segredo |

Se a maioria das detecções de uma categoria for menção, a prevalência daquela categoria está **superestimada**. Precisamos medir a **precisão** de cada detector: de tudo que ele marcou, quanto é instrução de verdade.

**Por que uma amostra:** não dá para ler 1,6 milhão de skills à mão. Sorteamos aleatoriamente alguns trechos detectados em cada categoria, rotulamos esses à mão e usamos a proporção correta na amostra como **estimativa** da precisão do detector, com intervalo de confiança. É o procedimento padrão para validar heurísticas em estudos de mineração de repositórios.

**Etapas:** 5.1 sortear a amostra → 5.2 rotular à mão (trabalho humano, fora do notebook) → 5.3 calcular a precisão → 5.4 medir a concordância entre dois avaliadores → 5.5 ver o efeito nos números do notebook 04.

**Situação da rotulagem:** completa. `label` é a pré-rotulagem feita com apoio de um modelo de linguagem (avaliador 1), `label_2` é a rotulagem humana (avaliador 2) e `label_final` é o consenso depois de discutir as divergências. A precisão usa `label_final`; o κ compara `label` com `label_2`, antes da discussão. A coluna `rater2_blind` marca os 43 trechos que o avaliador 2 rotulou sem ver a `label`.

---

### English summary
Regular expressions detect a *mention*, not an *instruction*. We draw a stratified random sample (seed 42) of matched snippets per capability and label each by hand as `instructs`, `mentions` or `false_match`. Precision = `instructs` / labelled, reported with 95% Wilson intervals; inter-rater agreement with Cohen's kappa.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("GITSKILLS_DB", str(ROOT / "data/raw/agent_skills_release.db"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
from gitskills import clean, db, reuse, adoption, patterns
from gitskills.plots import save, save_table
%matplotlib inline
print("database:", os.environ["GITSKILLS_DB"])
import re
from statsmodels.stats.proportion import proportion_confint

database: /Users/cleber/Documents/gitskills-mc/code/data/raw/agent_skills_release.db


## 5.1 Sortear a amostra
**Objetivo:** para cada uma das 9 categorias, sortear 12 skills em que a regex disparou (108 trechos no total) e guardar ~160 caracteres antes e depois do ponto detectado, para dar contexto a quem rotula. A semente fixa (42) faz o sorteio ser reproduzível. O arquivo só é criado se ainda não existir, para nunca apagar rótulos já feitos.

In [2]:
PER_CATEGORY = 12   # 9 categories x 12 = 108 snippets
skills = clean.strict(clean.load_skills())
rows = []
for cap, rx in patterns.COMPILED_RISKS.items():
    hits = skills[skills[f"risk_{cap}"].fillna(False)]
    pick = hits.sample(min(PER_CATEGORY, len(hits)), random_state=42)
    texts = db.query(f"SELECT file_sha, content FROM artifacts WHERE dedup_primary = 1 AND file_sha IN ({','.join('?' * len(pick))})",
                     tuple(pick["file_sha"]))
    for _, t in texts.iterrows():
        m = rx.search(t["content"] or "")
        if m:
            s = max(0, m.start() - 160); e = min(len(t["content"]), m.end() + 160)
            rows.append({"capability": cap, "file_sha": t["file_sha"], "snippet": t["content"][s:e].replace("\n", " "), "label": "", "label_2": ""})
sample = pd.DataFrame(rows)
out = ROOT / "results/tables/validation_sample.csv"
if not out.exists():
    sample.to_csv(out, index=False)
print(f"{len(sample)} snippets -> {out}")
sample.groupby("capability").size().rename("snippets").to_frame()

108 snippets -> /Users/cleber/Documents/gitskills-mc/code/results/tables/validation_sample.csv


,snippets
capability,
destructive_fs,12
dynamic_exec,12
git_push,12
package_install,12
pipe_to_shell,12
privilege,12
remote_fetch,12
safety_bypass,12
secrets,12


## 5.2 Como rotular (trabalho manual)
Abra `results/tables/validation_sample.csv` (Excel, Numbers ou Google Sheets), leia o `snippet` de cada linha e preencha a coluna **`label`** com uma de três palavras:

* **`instructs`**: o texto manda o agente executar a ação ("rode `curl … | bash`", "leia o arquivo `.env`");
* **`mentions`**: o texto fala da ação sem mandar executar ("nunca exponha a API key", "evite `rm -rf`");
* **`false_match`**: a regex casou com outra coisa ("token" no sentido de LLM, `--force` de outra ferramenta).

**Segundo avaliador (recomendado):** outra pessoa preenche a coluna **`label_2`** de forma independente, sem ver a `label`. Isso mede se o critério é claro (5.4). Depois as divergências são discutidas e a `label` final fica com o consenso.

Salve como CSV, sem mudar os nomes das colunas, e rode o notebook de novo.

## 5.3 Precisão por categoria
**Objetivo:** estimar, para cada categoria, a fração de detecções que são instruções reais, com intervalo de confiança de Wilson a 95%.

In [3]:
VALID = {"instructs", "mentions", "false_match"}
lab = pd.read_csv(ROOT / "results/tables/validation_sample.csv", dtype=str, keep_default_na=False)
if "label_2" not in lab:
    lab["label_2"] = ""
lab["label"] = lab["label"].str.strip().str.lower()
lab["label_2"] = lab["label_2"].str.strip().str.lower()
final = "label_final" if "label_final" in lab and (lab["label_final"] != "").all() else "label"
print("precision computed from:", final)
bad = lab[(lab["label"] != "") & ~lab["label"].isin(VALID)]
done = lab[lab["label"].isin(VALID)]
print(f"labelled: {len(done)}/{len(lab)} snippets" + (f"; {len(bad)} with an unknown label: {sorted(bad['label'].unique())}" if len(bad) else ""))
if done.empty:
    print("STATUS: waiting for manual labels (see 5.2). Precision cannot be computed yet.")
    prec = None
else:
    g = done.groupby("capability")[final]
    prec = pd.DataFrame({"labelled": g.size(), "instructs": g.apply(lambda s: int((s == "instructs").sum()))})
    prec["precision"] = prec["instructs"] / prec["labelled"]
    ci = [proportion_confint(k, n, method="wilson") for k, n in zip(prec["instructs"], prec["labelled"])]
    prec["ci_low"], prec["ci_high"] = [c[0] for c in ci], [c[1] for c in ci]
    save_table(prec.reset_index(), "validation_precision")
    display(prec.round(2))

precision computed from: label_final
labelled: 108/108 snippets


  table  -> results/tables/validation_precision.csv


,labelled,instructs,precision,ci_low,ci_high
capability,,,,,
destructive_fs,12,10,0.83,0.55,0.95
dynamic_exec,12,1,0.08,0.01,0.35
git_push,12,7,0.58,0.32,0.81
package_install,12,10,0.83,0.55,0.95
pipe_to_shell,12,12,1.00,0.76,1.00
privilege,12,9,0.75,0.47,0.91
remote_fetch,12,6,0.50,0.25,0.75
safety_bypass,12,4,0.33,0.14,0.61
secrets,12,2,0.17,0.05,0.45


**Como ler:** precisão 0,90 quer dizer que 9 em cada 10 detecções daquela categoria são instruções reais. Com 12 trechos por categoria, o intervalo de confiança é largo (cerca de ±25 pontos percentuais): serve para separar detectores bons de ruins, não para afinar a segunda casa decimal. Se uma categoria ficar abaixo de ~0,7, o número dela no notebook 04 deve ser reportado com ressalva, ou o detector deve ser refinado em `patterns.py`.

## 5.4 Concordância entre avaliadores (κ de Cohen)
**Objetivo:** verificar se duas pessoas, rotulando de forma independente, chegam ao mesmo rótulo. κ desconta a concordância que aconteceria por acaso: acima de 0,6 é considerado substancial, acima de 0,8 quase perfeito.

In [4]:
both = lab[lab["label"].isin(VALID) & lab["label_2"].isin(VALID)]
if both.empty:
    print("STATUS: no snippet has both 'label' and 'label_2' yet; kappa needs a second rater.")
else:
    cats = sorted(VALID)
    po = (both["label"] == both["label_2"]).mean()
    pe = sum((both["label"] == c).mean() * (both["label_2"] == c).mean() for c in cats)
    kappa = (po - pe) / (1 - pe) if pe < 1 else float("nan")
    print(f"snippets rated twice: {len(both)}; raw agreement {po:.1%}; Cohen's kappa {kappa:.2f}")
    display(pd.crosstab(both["label"], both["label_2"]))
    def kappa_bin(df):
        a, b = df["label"] == "instructs", df["label_2"] == "instructs"
        po = (a == b).mean(); pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
        return po, (po - pe) / (1 - pe)
    rows = [("all snippets", both)]
    if "rater2_blind" in both:
        rows.append(("rater 2 blind to label", both[both["rater2_blind"].astype(str) == "True"]))
    display(pd.DataFrame([(n, len(df), *kappa_bin(df)) for n, df in rows],
                         columns=["subset", "snippets", "agreement_instructs_vs_not", "kappa_instructs_vs_not"]).round(2))

snippets rated twice: 108; raw agreement 80.6%; Cohen's kappa 0.66


label_2,false_match,instructs,mentions
label,,,
false_match,13,3,0
instructs,1,56,3
mentions,8,6,18


,subset,snippets,agreement_instructs_vs_not,kappa_instructs_vs_not
0,all snippets,108,0.88,0.75
1,rater 2 blind to label,43,0.77,0.54


**O que os dados mostram:** nas três classes, κ = **0,66** (substancial). Quase todas as divergências separam `mentions` de `false_match`, por exemplo uma proibição como "never skip hooks", e as duas classes contam como **não** instrução, então não afetam a precisão. Na decisão que importa para a precisão (`instructs` ou não), κ = **0,75** nos 108 trechos e **0,54** (moderado) nos 43 rotulados às cegas. As 13 divergências binárias foram discutidas, e o consenso está em `label_final` (`validation_adjudication.xlsx`); ele mudou só uma precisão, a de `remote_fetch` (5/12 → 6/12).

## 5.5 Efeito nos números do notebook 04
**Objetivo:** mostrar o que a precisão muda na prevalência. Prevalência ajustada ≈ prevalência detectada × precisão. Por exemplo, se 17,6% das skills disparam `secrets` e só metade das detecções é instrução real, a prevalência real de instruções que lidam com segredos é ~9%.

In [5]:
strict = skills
detected = pd.Series({c: strict[f"risk_{c}"].mean() for c in patterns.COMPILED_RISKS}, name="detected_share")
if prec is None:
    print("STATUS: waiting for labels; showing detected prevalence only.")
    display(detected.to_frame().round(3))
else:
    adj = detected.to_frame().join(prec[["precision", "ci_low", "ci_high"]])
    adj["adjusted_share"] = adj["detected_share"] * adj["precision"]
    adj["adjusted_low"] = adj["detected_share"] * adj["ci_low"]
    adj["adjusted_high"] = adj["detected_share"] * adj["ci_high"]
    display(adj.round(3))

,detected_share,precision,ci_low,ci_high,adjusted_share,adjusted_low,adjusted_high
pipe_to_shell,0.007,1.000,0.758,1.000,0.007,0.005,0.007
remote_fetch,0.086,0.500,0.254,0.746,0.043,0.022,0.064
destructive_fs,0.011,0.833,0.552,0.953,0.009,0.006,0.011
privilege,0.021,0.750,0.468,0.911,0.016,0.010,0.019
dynamic_exec,0.044,0.083,0.015,0.354,0.004,0.001,0.016
package_install,0.119,0.833,0.552,0.953,0.100,0.066,0.114
secrets,0.176,0.167,0.047,0.448,0.029,0.008,0.079
git_push,0.038,0.583,0.320,0.807,0.022,0.012,0.030
safety_bypass,0.007,0.333,0.138,0.609,0.002,0.001,0.004


## Conclusão do notebook 05
Este é o passo de **validação** da metodologia: transforma "a regex disparou" em uma estimativa de quanto confiar em cada número da RQ5. Sem ele, os resultados do notebook 04 devem ser lidos como **limites superiores** (detecções, não instruções confirmadas). Com ele, cada prevalência pode ser reportada com a precisão do detector e, se houver dois avaliadores, com a concordância κ.